# Stored execution evidence — Plant Pathology 2020

This public evidence copy preserves every textual output cell supplied in `jiaozi.zip`.
The original notebook SHA-256 is `5914bac3ce9a1cf40e3b2d28d7c828cbc22cf7dfa19a3c570f8f1ae82d96acfe`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Integration Update on Colab

This notebook deletes any existing `/content/Jiaozi` checkout, clones the latest `main` branch, installs dependencies, and runs the integrated Jiaozi pipeline end to end on Colab.

The active flow is:

1. Module 1 parses the natural-language task with an LLM.
2. Module 2 analyzes the HuggingFace dataset.
3. Module 3 retrieves ranked CV model candidates.
4. Module 4 generates runnable training code.
5. The generated project runs the built-in smoke checks.


## Before running

1. In Colab, open **Secrets** and add `OPENAI_API_KEY`.
2. Add `KAGGLE_API_TOKEN` in Colab Secrets and accept the Plant Pathology competition rules on Kaggle before downloading.
3. Select a GPU runtime for real training.
4. Run the cells in order.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

def normalize_repo_url(value: str) -> str:
    value = (value or '').strip()
    markdown_match = re.fullmatch(r'\[(.*?)\]\((https?://[^)]+)\)', value)
    if markdown_match:
        return markdown_match.group(2)
    plain_url_match = re.search(r'https?://\S+', value)
    if plain_url_match:
        return plain_url_match.group(0)
    return value

REPO_URL = normalize_repo_url(os.getenv('JIAOZI_REPO_URL', 'https://github.com/Isso-W/Jiaozi.git'))
REPO_REF = os.getenv('JIAOZI_REPO_REF', 'main')
REPO_DIR = Path('/content/Jiaozi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print("REPO_URL =", REPO_URL)
print("REPO_REF =", REPO_REF)

clone_cmd = [
    'git', 'clone',
    '--depth', '1',
    '--branch', REPO_REF,
    REPO_URL,
    str(REPO_DIR),
]

completed = subprocess.run(
    clone_cmd,
    capture_output=True,
    text=True,
)

print("=== git stdout ===")
print(completed.stdout)
print("=== git stderr ===")
print(completed.stderr)

completed.check_returncode()

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("\n=== requirements.txt ===")
print(Path("requirements.txt").read_text()[:4000])

print("\n=== installing requirements ===")
pip_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
    capture_output=True,
    text=True,
)

print(pip_result.stdout)
print(pip_result.stderr)
print("pip return code =", pip_result.returncode)

if pip_result.returncode != 0:
    raise RuntimeError("pip install failed. Check the pip output above.")

print('Jiaozi repository:', REPO_DIR)
print('Jiaozi repo URL:', REPO_URL)
print('Jiaozi ref:', REPO_REF)

pipeline_candidates = list(REPO_DIR.rglob("pipeline.py"))
print("pipeline.py candidates:")
for p in pipeline_candidates:
    print(" -", p)


REPO_URL = https://github.com/muzhi-hac/Jiaozi-rag-wang.git
REPO_REF = rag_wang
=== git stdout ===

=== git stderr ===
Cloning into '/content/Jiaozi'...


=== requirements.txt ===
# Runtime dependencies for the Jiaozi CV Auto-DL pipeline.
# Install with the same Python interpreter used to run tests, for example:
#   /usr/bin/python3 -m pip install -r requirements.txt

chromadb
datasets
networkx
numpy
openai>=2.0.0
pandas
pillow
scikit-learn
sentence-transformers
torch
torchvision
transformers

# Test runner for the module4_agent pytest suite.
pytest


=== installing requirements ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 138.8 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-5.5')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

del openai_api_key
del kaggle_token


Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token


## Download the Plant Pathology data

These cells configure the Kaggle CLI, download `plant-pathology-2020-fgvc7`, and extract the competition files under `/content/data`. The pipeline cell later converts the one-hot `train.csv` into a single `label` column that Jiaozi's local CSV loader can consume.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

def run_pip(args):
    print("Running:", " ".join([sys.executable, "-m", "pip"] + args))
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + args,
        capture_output=True,
        text=True,
    )
    print("=== pip stdout ===")
    print(result.stdout)
    print("=== pip stderr ===")
    print(result.stderr)
    print("return code =", result.returncode)
    return result

# First upgrade the basic installation tools
run_pip(["install", "--upgrade", "pip", "setuptools", "wheel"])

# Install the Kaggle CLI again, but do not use the -q option.
kaggle_install = run_pip(["install", "--upgrade", "kaggle"])

if kaggle_install.returncode != 0:
    raise RuntimeError("Kaggle CLI install failed. Check pip output above.")

# Read Kaggle access token from Colab Secrets
kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise RuntimeError("Please add KAGGLE_API_TOKEN in Colab Secrets.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token.strip(), encoding="utf-8")
access_token_path.chmod(0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

print("Kaggle access token configured.")

# Check if Kaggle is available
subprocess.run(["kaggle", "--version"], check=False)
##%%
KAGGLE_COMPETITION = "plant-pathology-2020-fgvc7"


In [ ]:
from pathlib import Path
import zipfile

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_files = sorted(DATA_ROOT.glob(f"{KAGGLE_COMPETITION}*.zip")) or sorted(DATA_ROOT.glob("*.zip"))
print("Zip files:", zip_files)

if not zip_files:
    raise FileNotFoundError("No .zip file found in /content/data")

zip_path = zip_files[0]
KAGGLE_DATA_DIR = DATA_ROOT / zip_path.stem
KAGGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting:", zip_path)
print("To:", KAGGLE_DATA_DIR)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(KAGGLE_DATA_DIR)

# Some Kaggle competitions contain nested archives. Extract them in place so image
# discovery below can find the real files regardless of packaging.
for nested_zip in sorted(KAGGLE_DATA_DIR.rglob("*.zip")):
    nested_target = nested_zip.with_suffix("")
    nested_target.mkdir(parents=True, exist_ok=True)
    print("Extracting nested:", nested_zip, "->", nested_target)
    with zipfile.ZipFile(nested_zip, "r") as z:
        z.extractall(nested_target)

print("Done.")
print("KAGGLE_DATA_DIR =", KAGGLE_DATA_DIR)
print("Top-level files:")
for p in sorted(KAGGLE_DATA_DIR.iterdir()):
    print(" -", p.name)

#%% md


## Mount Google Drive for caches and checkpoints

HuggingFace caches and training checkpoints can live on Drive so a recycled Colab runtime does not force every artifact to be downloaded or trained from scratch.


In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/drive/MyDrive/Jiaozi/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')


Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/Jiaozi/hf_cache
First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).
Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.


## Run the integrated pipeline

This cell runs the integrated Jiaozi flow against the local Kaggle files:

1. Convert Plant Pathology's one-hot labels into a local CSV with `image_id,label`.
2. Run Module 1 from the natural-language `QUERY`.
3. Run Module 2 on sampled real images from the Kaggle folder.
4. Run Module 3 retrieval and optional recommender re-ranking.
5. Run Module 4 code generation with local CSV training metadata.

Set `REAL_TRAINING = True` to generate real-training code and train it in the next cell. With `REAL_TRAINING = False`, Module 4 keeps the generated project in smoke-test mode.


In [8]:
import json
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

REPO_DIR = Path("/content/Jiaozi")
PIPELINE_PATH = REPO_DIR / "pipeline.py"

if not PIPELINE_PATH.exists():
    raise FileNotFoundError(
        f"No pipeline.py found at {PIPELINE_PATH}. Please rerun the git clone cell first."
    )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Current working directory:", Path.cwd())
print("Using pipeline:", PIPELINE_PATH)

# User input: describe the task naturally and let Module 3 choose the model.
QUERY = (
    "Classify plant leaf images from the Plant Pathology Kaggle dataset to recognize "
    "healthy, multiple diseases, rust, and scab. Accuracy and mean ROC AUC matter more than speed."
)

REAL_TRAINING = True
EPOCHS = None
OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")

DATASET_DIR = Path(KAGGLE_DATA_DIR)
raw_train_csvs = sorted(DATASET_DIR.rglob("train.csv"))
if not raw_train_csvs:
    raise FileNotFoundError(f"No train.csv found under {DATASET_DIR}")
raw_train_csv = raw_train_csvs[0]

raw_frame = pd.read_csv(raw_train_csv)
image_column = "image_id" if "image_id" in raw_frame.columns else "image"
if image_column not in raw_frame.columns:
    raise ValueError(f"Could not find an image id column in {raw_train_csv}: {list(raw_frame.columns)}")

known_label_columns = ["healthy", "multiple_diseases", "rust", "scab"]
label_columns = [column for column in known_label_columns if column in raw_frame.columns]
if not label_columns and "label" not in raw_frame.columns:
    ignored = {image_column, "id", "image", "image_id"}
    label_columns = [
        column for column in raw_frame.columns
        if column not in ignored and pd.api.types.is_numeric_dtype(raw_frame[column])
    ]
if "label" in raw_frame.columns:
    processed_frame = raw_frame[[image_column, "label"]].copy()
else:
    if not label_columns:
        raise ValueError(f"Could not infer label columns from {raw_train_csv}: {list(raw_frame.columns)}")
    if (raw_frame[label_columns].sum(axis=1) <= 0).any():
        raise ValueError("At least one training row has no positive label.")
    processed_frame = raw_frame[[image_column, *label_columns]].copy()
    processed_frame["label"] = processed_frame[label_columns].idxmax(axis=1)
    processed_frame = processed_frame[[image_column, "label"]]

processed_train_csv = raw_train_csv.with_name("jiaozi_train.csv")
processed_frame.to_csv(processed_train_csv, index=False)

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
sample_image_id = str(processed_frame.iloc[0][image_column])
sample_stem = Path(sample_image_id).stem
matches = [
    path for path in DATASET_DIR.rglob(f"{sample_stem}*")
    if path.is_file()
    and path.suffix.lower() in image_exts
    and (path.stem == sample_stem or path.name == sample_image_id)
]
if not matches:
    raise FileNotFoundError(f"Could not locate an image file for sample id {sample_image_id!r} under {DATASET_DIR}")
image_dir = matches[0].parent
image_extension = "" if Path(sample_image_id).suffix else matches[0].suffix

LOCAL_DATA_INFO = {
    "benchmark": "plant_pathology_2020",
    "competition": "plant-pathology-2020-fgvc7",
    "train_csv": str(processed_train_csv),
    "image_dir": str(image_dir),
    "image_column": image_column,
    "label_column": "label",
    "image_path_template": "{image}",
    "image_extension": image_extension,
    "num_classes": int(processed_frame["label"].nunique()),
    "metric": "roc_auc",
}

print("Prepared local Kaggle data:")
print(json.dumps(LOCAL_DATA_INFO, indent=2, ensure_ascii=False))
print("Class distribution:")
print(processed_frame["label"].value_counts().to_string())

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

from analyzer.image_statistics import ImageStatisticsAnalyzer
from features_extraction_api import module1_pipeline
from pipeline import derive_recommended_epochs, merge_modules, run_module4_generation
from recommender import OutcomeMemory, recommend
from retrieval.rag_retrieval import (
    build_all_task_lists,
    build_graph,
    build_vector_index,
    print_results,
    retrieve_top3_hybrid,
)

print("[Notebook] Module 1: Parsing user requirements...")
m1_output = module1_pipeline(QUERY)
if m1_output is None:
    raise RuntimeError("Module 1 failed, so no Module 3 or Module 4 output was produced.")

class LocalImageSplit:
    column_names = ["image", "label"]

    def __init__(self, frame, info):
        from PIL import Image

        self._Image = Image
        self.labels = frame[info["label_column"]].tolist()
        self.image_paths = []
        image_root = Path(info["image_dir"])
        image_column = info["image_column"]
        label_column = info["label_column"]
        template = info.get("image_path_template", "{image}")
        extension = info.get("image_extension", "")

        for _, row in frame.iterrows():
            image_value = str(row[image_column])
            relative = template.format(
                image=image_value,
                label=str(row[label_column]),
                stem=Path(image_value).stem,
            )
            image_path = image_root / relative
            if extension and not image_path.suffix:
                image_path = image_path.with_suffix(extension)
            if not image_path.is_file():
                raise FileNotFoundError(f"Training image referenced by CSV is missing: {image_path}")
            self.image_paths.append(image_path)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, key):
        if key == "label":
            return self.labels
        if key == "image":
            return [self._open_image(path) for path in self.image_paths]
        index = int(key)
        return {"image": self._open_image(self.image_paths[index]), "label": self.labels[index]}

    def _open_image(self, path):
        with self._Image.open(path) as image:
            return image.convert("RGB")


def build_local_module2_dataset(info):
    frame = pd.read_csv(info["train_csv"])
    required = {info["image_column"], info["label_column"]}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Missing required CSV columns: {sorted(missing)}")
    return {"train": LocalImageSplit(frame, info)}


print("[Notebook] Module 2: Analyzing sampled real Kaggle images...")
m2_report = ImageStatisticsAnalyzer().analyze(build_local_module2_dataset(LOCAL_DATA_INFO))

m3_input = merge_modules(m1_output, m2_report)
m3_input["evaluation_metric"] = "roc_auc"
print(
    f"[Notebook] Merged: task={m3_input['task_type']} "
    f"size={m3_input['data_size']} classes={m3_input.get('num_classes')} "
    f"metric={m3_input['evaluation_metric']}"
)

print("[Notebook] Module 3: Retrieving model configurations...")
graph = build_graph()
collection = build_vector_index()
recommendations = retrieve_top3_hybrid(m3_input, graph, collection)
print_results(m3_input, recommendations, graph)

try:
    recommendations = recommend(recommendations, m2_report, m3_input, memory=OutcomeMemory())
    print("[Notebook] Recommender re-ranked candidates.")
except Exception as exc:
    print(f"[Notebook] Recommender skipped: {exc}")

task_lists = build_all_task_lists(recommendations, graph, fmt="nl")
for task_list in task_lists:
    model_config = task_list.get("model_config")
    if not isinstance(model_config, dict):
        continue
    model_config.update(
        {
            "num_classes": LOCAL_DATA_INFO["num_classes"],
            "train_csv": LOCAL_DATA_INFO["train_csv"],
            "image_dir": LOCAL_DATA_INFO["image_dir"],
            "image_column": LOCAL_DATA_INFO["image_column"],
            "label_column": LOCAL_DATA_INFO["label_column"],
            "image_path_template": LOCAL_DATA_INFO["image_path_template"],
            "image_extension": LOCAL_DATA_INFO["image_extension"],
            "evaluation_metric": LOCAL_DATA_INFO["metric"],
            "offline_smoke": not REAL_TRAINING,
            "benchmark_key": LOCAL_DATA_INFO["benchmark"],
            "competition": LOCAL_DATA_INFO["competition"],
        }
    )
    model_config.setdefault(
        "recommended_epochs",
        derive_recommended_epochs(
            m3_input.get("data_size", "medium"),
            model_config.get("finetune_strategy"),
            bool(model_config.get("pretrained_hf_id")),
        ),
    )

module4 = run_module4_generation(
    task_lists,
    OUTPUT_DIR,
    skip_smoke=REAL_TRAINING,
    timeout=120,
    llm_provider="openai",
)

summary_path = OUTPUT_DIR / "module4_summary.json"
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    raise FileNotFoundError(
        f"Module 4 summary was not generated at {summary_path}. Available files: {available}"
    )

DATASET = LOCAL_DATA_INFO["train_csv"]
print("Module 4 summary:", summary_path)


Current working directory: /content/Jiaozi
Using pipeline: /content/Jiaozi/pipeline.py
Prepared local Kaggle data:
{
  "benchmark": "plant_pathology_2020",
  "competition": "plant-pathology-2020-fgvc7",
  "train_csv": "/content/data/plant-pathology-2020-fgvc7/jiaozi_train.csv",
  "image_dir": "/content/data/plant-pathology-2020-fgvc7/images",
  "image_column": "image_id",
  "label_column": "label",
  "image_path_template": "{image}",
  "image_extension": ".jpg",
  "num_classes": 4,
  "metric": "roc_auc"
}
Class distribution:
label
rust                 622
scab                 592
healthy              516
multiple_diseases     91
[Notebook] Module 1: Parsing user requirements...
[Notebook] Module 2: Analyzing sampled real Kaggle images...
[Notebook] Merged: task=classification size=small classes=4 metric=roc_auc
[Notebook] Module 3: Retrieving model configurations...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Index] Added 18 backbone entries and refreshed 18 total entries.

──────────────────────────────────────────────────────────────────────
Task   : classification
Input  : size=small  priority=accuracy
Desc   : Classify plant leaf images from the Plant Pathology Kaggle dataset to recognize healthy, multiple diseases, rust, and scab. Accuracy and mean ROC AUC matter more than speed.
──────────────────────────────────────────────────────────────────────

Top 1  [score=0.945  struct=1.0  vec=0.862]
  backbone  : DINOv3
  head      : Classification Head
  loss      : CrossEntropyLoss
  optimizer : AdamW
  pretrained: DINOv3-B/16 / LVD-1689M
               facebook/dinov3-vitb16-pretrain-lvd1689m  (LVD-1689M (self-supervised), 86M)
               strategy: partial finetune last 2 blocks
  training  : pretrained weights required (insufficient data to train from scratch)

Top 2  [score=0.704  struct=0.786  vec=0.582]
  backbone  : DINOv2
  head      : Classification Head
  loss      : CrossEnt

In [9]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')


{
  "candidates": [
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 1,
      "score": 0.945,
      "task_type": "classification"
    },
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 2,
      "score": 0.945,
      "task_type": "classification"
    },
    {
      "backbone": "dinov2",
      "finetune_strategy": "head_only",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 3,
      "score": 0.704,
      "task_type": "classification"
    },
    {
      "backbone": "efficientnet",
      "finetune_strategy": "head_only",
      "loss": "cross_entropy_loss",
      "optimizer": "adam",
      "rank": 4,
      "score": 0.629,
      "task_type": "classification"
    }
  ],
  "errors": [],
  "generated_files": [
    "README_generated.md",
    "configs.json",
    "eval

## Train the recommended model (real data)

This step only runs when `REAL_TRAINING = True`. It executes the generated `run.py`,
which loads the real dataset, trains the model Module 3 recommended, evaluates on the
test split, and saves checkpoints. Select a GPU runtime first for reasonable speed.


In [10]:
import json
import os
import subprocess
import sys
from pathlib import Path

if not REAL_TRAINING:
    print('REAL_TRAINING is False - skipping the real training step.')
    print('Set REAL_TRAINING = True in the run cell above to train on real data.')
else:
    # Persist checkpoints on Google Drive so they survive runtime recycling, and
    # resume automatically if a previous run left one. Set False to keep them in
    # the ephemeral generated project dir instead.
    SAVE_CHECKPOINTS_TO_DRIVE = True

    cfg_path = OUTPUT_DIR / 'configs.json'
    configs = json.loads(cfg_path.read_text(encoding='utf-8'))
    cfg = configs[0]
    mc = cfg.get('model_config', cfg)

    epochs = EPOCHS
    if epochs is None:
        epochs = mc.get('recommended_epochs') or cfg.get('recommended_epochs') or 10
    backbone = mc.get('backbone') or cfg.get('backbone')
    dataset_used = mc.get('dataset_id') or cfg.get('dataset_id') or DATASET

    if SAVE_CHECKPOINTS_TO_DRIVE and Path('/content/drive/MyDrive').exists():
        tag = f"{backbone}_{str(dataset_used).replace('/', '_')}"
        ckpt_dir = f'/content/drive/MyDrive/Jiaozi/checkpoints/{tag}'
        os.makedirs(ckpt_dir, exist_ok=True)
        cfg['checkpoint_dir'] = ckpt_dir
        cfg['resume_checkpoint'] = 'auto'
        cfg_path.write_text(json.dumps(configs, indent=2, ensure_ascii=False), encoding='utf-8')
        print('Checkpoints ->', ckpt_dir, '(resume: auto)')
    elif SAVE_CHECKPOINTS_TO_DRIVE:
        print('WARNING: Drive not mounted (run the Drive cell above); checkpoints will be ephemeral.')

    print(f'Training {backbone} for {epochs} epochs on {dataset_used} ...')
    train_command = [sys.executable, '-u', 'run.py', '--epochs', str(epochs)]
    print('Command:', ' '.join(train_command), '(cwd:', OUTPUT_DIR, ')\n')
    subprocess.run(train_command, cwd=str(OUTPUT_DIR), text=True).check_returncode()
    print('\nReal training finished. Checkpoints under:', cfg.get('checkpoint_dir', 'checkpoints (ephemeral)'))


Checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/dinov3__content_data_plant-pathology-2020-fgvc7_jiaozi_train.csv (resume: auto)
Training dinov3 for 40 epochs on /content/data/plant-pathology-2020-fgvc7/jiaozi_train.csv ...
Command: /usr/bin/python3 -u run.py --epochs 40 (cwd: /content/jiaozi_generated_pipeline )


Real training finished. Checkpoints under: /content/drive/MyDrive/Jiaozi/checkpoints/dinov3__content_data_plant-pathology-2020-fgvc7_jiaozi_train.csv


## Show the result

Reads the best checkpoint and prints its validation score **instantly** (the metric was
already computed during training — no re-evaluation). Set `FULL_EVAL = True` for a full
re-evaluation (macro-F1, params, sample count), which costs one extra pass over the eval set.


In [11]:
import json, torch
from pathlib import Path

# Locate best_model.pt (prefer the Drive checkpoint dir set by the training cell).
_candidates = []
try:
    _candidates.append(Path(ckpt_dir) / 'best_model.pt')
except NameError:
    pass
_candidates.append(Path(OUTPUT_DIR) / 'checkpoints' / 'best_model.pt')
best_path = next((p for p in _candidates if p.exists()), None)
if best_path is None:
    raise FileNotFoundError(f'best_model.pt not found in: {[str(p) for p in _candidates]}')

ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
print('=== RESULT (best checkpoint) ===')
print('checkpoint :', best_path)
print('best_epoch :', ckpt.get('best_epoch'))
print('best_metric:', ckpt.get('best_metric'), '(validation metric at best epoch — no re-eval)')
print('validation :', json.dumps(ckpt.get('validation'), ensure_ascii=False))

# Full re-evaluation (macro_f1, params, num_samples) — one extra pass over the eval set.
FULL_EVAL = False
if FULL_EVAL:
    import os, sys
    os.chdir(OUTPUT_DIR); sys.path.insert(0, str(OUTPUT_DIR))
    from model import build_model
    from evaluate import evaluate
    _cfg = json.loads((Path(OUTPUT_DIR) / 'configs.json').read_text(encoding='utf-8'))[0]
    _cfg.update(_cfg.pop('model_config', {}) or {})
    _model = build_model(_cfg); _model.load_state_dict(ckpt['model_state_dict'])
    print('\n=== FULL EVALUATE ===')
    print(json.dumps(evaluate(_model, _cfg), indent=2, ensure_ascii=False))


=== RESULT (best checkpoint) ===
checkpoint : /content/drive/MyDrive/Jiaozi/checkpoints/dinov3__content_data_plant-pathology-2020-fgvc7_jiaozi_train.csv/best_model.pt
best_epoch : 40
best_metric: 0.976634685523388 (validation metric at best epoch — no re-eval)
validation : {"metric_name": "roc_auc", "metric_value": 0.976634685523388, "accuracy": 0.9146005511283875, "epoch": 40}


In [12]:
import os
import json
import torch
import pandas as pd
from pathlib import Path
from PIL import Image

OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")
DATA_DIR = Path("/content/data/plant-pathology-2020-fgvc7")

sample_path = DATA_DIR / "sample_submission.csv"
test_csv = DATA_DIR / "test.csv"
image_dir = DATA_DIR / "images"
submission_path = OUTPUT_DIR / "submission.csv"

assert sample_path.exists(), sample_path
assert test_csv.exists(), test_csv
assert image_dir.exists(), image_dir

os.chdir(OUTPUT_DIR)
if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))

from model import build_model
from train import _build_image_transform

configs = json.loads((OUTPUT_DIR / "configs.json").read_text(encoding="utf-8"))
cfg = configs[0]
if "model_config" in cfg:
    flat_cfg = {**cfg, **cfg["model_config"]}
else:
    flat_cfg = cfg

ckpt_candidates = []
try:
    ckpt_candidates.append(Path(ckpt_dir) / "best_model.pt")
except NameError:
    pass
ckpt_candidates.append(Path(flat_cfg.get("checkpoint_dir", "checkpoints")) / "best_model.pt")
ckpt_candidates.append(OUTPUT_DIR / "checkpoints" / "best_model.pt")

best_path = next((p for p in ckpt_candidates if p.exists()), None)
if best_path is None:
    raise FileNotFoundError(f"best_model.pt not found in: {[str(p) for p in ckpt_candidates]}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(flat_cfg).to(device)
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

transform = _build_image_transform(flat_cfg, "test")

sample = pd.read_csv(sample_path)
test_ids = sample["image_id"].astype(str).tolist()
class_cols = ["healthy", "multiple_diseases", "rust", "scab"]

rows = []
batch = []
batch_ids = []

def flush():
    global batch, batch_ids, rows
    if not batch:
        return
    x = torch.stack(batch).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1).cpu().numpy()
    for image_id, p in zip(batch_ids, probs):
        rows.append([image_id, *p.tolist()])
    batch = []
    batch_ids = []

for image_id in test_ids:
    img_path = image_dir / f"{image_id}.jpg"
    if not img_path.exists():
        raise FileNotFoundError(img_path)
    img = Image.open(img_path).convert("RGB")
    batch.append(transform(img))
    batch_ids.append(image_id)
    if len(batch) >= 64:
        flush()

flush()

sub = pd.DataFrame(rows, columns=["image_id", *class_cols])
sub = sample[["image_id"]].merge(sub, on="image_id", how="left")

# sanity checks
assert sub[class_cols].notna().all().all()
assert ((sub[class_cols] >= 0) & (sub[class_cols] <= 1)).all().all()
assert abs(sub[class_cols].sum(axis=1).sub(1)).max() < 1e-4

sub.to_csv(submission_path, index=False)
print("Wrote:", submission_path)
display(sub.head())


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wrote: /content/jiaozi_generated_pipeline/submission.csv


,image_id,healthy,multiple_diseases,rust,scab
0,Test_0,4.271935e-06,0.000626,0.999364,5.362394e-06
1,Test_1,1.515947e-06,0.055395,0.944187,4.165002e-04
2,Test_2,6.527573e-05,0.000193,0.000006,9.997351e-01
3,Test_3,9.991092e-01,0.000001,0.000094,7.957070e-04
4,Test_4,5.117315e-08,0.000178,0.999821,6.894422e-07


In [ ]:
  -f /content/jiaozi_generated_pipeline/submission.csv \
  -m "Jiaozi late submission"
#%%
!kaggle competitions submissions -c plant-pathology-2020-fgvc7
